# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the mlcroissant Dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset name and description
print("Dataset Title: {}".format(dataset.metadata.name))
print("Description:\n{}".format(dataset.metadata.description))

## 2. Data Overview
Review available record sets, fields, and their `@id`s in the dataset.

In [ ]:
# List all record sets by their @id and name
print("Available Record Sets (@id, name):")
for rs in dataset.metadata.record_sets:
    print(f"  {rs['@id']} - {rs.get('name', 'N/A')}")

# For demonstration, list the fields of all record sets
for rs in dataset.metadata.record_sets:
    print(f"\nRecord Set '{rs['@id']}' fields:")
    for field in rs['fields']:
        print(f"  {field['@id']} (name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to load all record sets into DataFrames
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")

# For further analysis, select the first available record set if unsure:
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nSampling head of '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may involve removing outliers, transforming distributions, or grouping data.

In [ ]:
# EDA on chosen/first record set
from pandas.api.types import is_numeric_dtype

# Ensure we have a main record set
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"Analyzing columns: {df.columns.tolist()}")
    # Try to pick a numeric field from the first record set
    numeric_fields = [col for col in df.columns if is_numeric_dtype(df[col])]
    if len(numeric_fields) == 0:
        print("No numeric fields found for EDA.")
    else:
        numeric_field_id = numeric_fields[0]
        print(f"Chosen numeric field for EDA: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        # Filtering
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered rows where {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[numeric_field_id + '_normalized'] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() or 1)
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())
        # Group by first non-numeric field, if available
        group_candidates = [col for col in df.columns if not is_numeric_dtype(df[col])]
        if len(group_candidates) > 0:
            group_field_id = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id} (mean):")
            display(grouped_df.head())
        else:
            print("No suitable string/categorical field for grouping.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution and its normalization, if available
if main_record_set_id and 'numeric_field_id' in locals():
    fig, axs = plt.subplots(1, 2, figsize=(12,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=axs[0])
    axs[0].set_title(f'Distribution of {numeric_field_id}')
    if numeric_field_id + '_normalized' in filtered_df.columns:
        sns.histplot(filtered_df[numeric_field_id + '_normalized'].dropna(), kde=True, ax=axs[1])
        axs[1].set_title(f'Normalized {numeric_field_id} (filtered)')
    else:
        axs[1].set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    print("No available numeric field for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.